#### Распространение ошибки при вычислении орбитальных элементов

In [1]:
from math import *
import numpy as np
from datetime import datetime
from astropy.time import Time

import matplotlib.pyplot as plt
%matplotlib widget

In [2]:
def rotx(psi):
    return np.array([[1,0,0],[0,cos(psi),sin(psi)],[0,-sin(psi),cos(psi)]])

def roty(psi):
    return np.array([[cos(psi),0,-sin(psi)],[0,1,0],[sin(psi),0,cos(psi)]])

def rotz(psi):
    return np.array([[cos(psi),sin(psi),0],[-sin(psi),cos(psi),0],[0,0,1]])


In [3]:
def orbital_elem(x,y,z,xdot,ydot,zdot):
    r = np.array([x,y,z])
    rdot = np.array([xdot,ydot,zdot])
    #module of r 
    R = sqrt(x*x+y*y+z*z)
    #radial velocity
    vr = np.dot(r/R,rdot)
    #double sectorial velocity
    c = np.array([y*zdot-z*ydot,z*xdot - x*zdot,x*ydot-y*xdot])
    C = np.linalg.norm(c)

    p = C**2 # focal parameter
    
    orb1 = p/R-1.0
    orb2 = vr*C
    # eccentricity
    e = sqrt(orb1*orb1+orb2*orb2)
    a = p/(1-e*e)

    u = atan2(orb2/e,orb1/e) # true anomaly
    # eccentric anomaly
    E = atan2(R*sin(u)/a/sqrt(1-e**2),R*cos(u)/a+e)
    M = E - e*sin(E) # mean anomaly
    P = 2*pi*a**(3/2) # period
    n = a**(-3/2) # mean motion
    T = - M / n # perihelion epoch
    #inclination
    i = atan2(c[2]/C,sqrt(c[0]**2+c[1]**2)/C)
    # longitude of the ascending node
    N = np.cross(np.array([0,0,1]),c/C)
#     print(N)
    Omega = atan2(N[1],N[0])
    omega_plus_u = atan2(np.dot(np.cross(N,r/R),c/C),np.dot(r/R,N))
    omega = omega_plus_u - u # argument of periapsis
    return a,e,P,T,i,Omega,omega

def KeplerEquation(e, P, T, t):
    n = 2 * pi / P
    E = M = n * (t - T)
    epsilon = pi/(180*3600000)
    while (fabs(E - e * sin(E) - M) > epsilon):
        E = M + e * sin(E)
    return E

def xyz(a,e,P,T,i,Omega, omega,t):   #heliocentric coordinates
    E = KeplerEquation(e,P,T,t)
    ksi = a*(cos(E)-e)
    eta = a*sqrt(1-e**2)*sin(E)
    r = np.array([ksi, eta, 0])
    return np.dot(rotz(-Omega),np.dot(rotx(-i),np.dot(rotz(-omega),r)))

In [4]:
x = -4.42886945; y = 0.48312824; z = 0.48312824 # au
xdot = -0.30709289; ydot = -0.27567732; zdot = -0.27567732 # в скоростях Земли

In [5]:
# проверка правильности работы кода вышенаписанных функций
a,e,P,T,i,Omega,omega = orbital_elem(x,y,z,xdot,ydot,zdot)
# вот мы посчитали орбитальные элементы
# теперь обратно сделаем из них вектор состояния и убедимся, что разности равны нулю (ну почти)...
xyz(a,e,P,T,i,Omega, omega,0) - np.array([x,y,z])

array([5.66580383e-09, 5.08619308e-09, 5.08619297e-09])

In [6]:
N = 1000
perr = 0.02
verr = 0.003
x = -4.42886945+np.random.normal(0,perr,N) 
y = 0.48312824+np.random.normal(0,perr,N) 
z = 0.48312824+np.random.normal(0,perr,N)
xdot = -0.30709289+np.random.normal(0,verr,N)
ydot = -0.27567732+np.random.normal(0,verr,N)
zdot = -0.27567732+np.random.normal(0,verr,N) 

orb_elements = []
for k in range(N):
    orb_elements.append(orbital_elem(x[k],y[k],z[k],xdot[k],ydot[k],zdot[k]))
orb_elements = np.asarray(orb_elements)

In [7]:
fig,ax = plt.subplots(1,figsize=(1.5, 1.5), dpi=300)
# a,e,P,T,i,Omega,omega
ax.scatter(orb_elements[:,0],orb_elements[:,1],  c='black',  s=0.3, linewidths=0.5)
ax.set_xlabel('a', fontsize = 6)
ax.set_ylabel('e', fontsize = 6)

ax.tick_params(axis='both', which='major', labelsize=4)
ax.tick_params(axis='both', which='minor', labelsize=4)
ax.ticklabel_format(useOffset=False)

plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …